# Phase 00 — Reproducibility Snapshot

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Capture the current legacy model, datasets, artifacts, reports, runtime environment, baseline metrics, and known risks before new experiments begin.

This notebook is the Phase 0 source of truth. It writes a reproducibility report to `reports/phase_00_reproducibility_snapshot.json` so later phases can compare experiments against the same snapshot.


## Contract boundary

Model/training owns only these core signals for the current training roadmap:

- `jobFitAlignment.score` and grounded alignment signals.
- `atsFriendliness.score` and detected ATS issues.
- `overallImpression` as grounded summary signal.

Backend/API wrapper owns actionables, section reviews, final job hydration, auth, persistence, request validation, and OpenAI wrapper orchestration.


## Shared setup

### Purpose
Define reusable paths and helper functions used by the Phase 0 steps.

### Required input
Repository root with the `legacy/`, `training/`, `reports/`, and `references/` directories available.

### Action
Load Python helpers only. No experiment training occurs in this notebook.

### Expected output
Reusable helper functions for file metadata, artifact inventory, environment capture, dataset counts, baseline metrics, and risk registration.

### Verification
The setup cell must run without mutating model or dataset artifacts. Any generated output must be limited to the Phase 0 report JSON.


In [7]:
from __future__ import annotations

import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "legacy").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd().resolve())
LEGACY = ROOT / "legacy"
REPORTS = ROOT / "reports"
PHASE0_REPORT = REPORTS / "phase_00_reproducibility_snapshot.json"

SPLIT_SEED = 42
VAL_SIZE = 0.15
TOLERANCE = 0.15
LABEL_BANDS = {
    "low": (0.00, 0.34),
    "medium": (0.35, 0.64),
    "high": (0.65, 1.00),
}

pd.set_option("display.max_colwidth", 120)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def file_metadata(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {"exists": False}
    stat = path.stat()
    return {
        "exists": True,
        "size_bytes": int(stat.st_size),
        "sha256": sha256_file(path),
        "modified_at_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
    }


def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not_installed"


def artifact_format(path: Path) -> str:
    if path.suffix:
        return path.suffix.lstrip(".").lower()
    return "directory" if path.is_dir() else "unknown"


def artifact_owner(path: Path) -> str:
    path_text = path.as_posix()
    if path_text.startswith("legacy/"):
        return "legacy-model-training"
    if path_text.startswith("training/notebooks/"):
        return "notebook-first-training"
    if path_text.startswith("reports/"):
        return "model-training-audit"
    return "model-training"


def artifact_role(path: Path) -> tuple[str, bool]:
    path_text = path.as_posix()
    inference_required = {
        "legacy/models/model_jobfit_v1.keras",
        "legacy/cache/all_job_embeddings.npy",
        "legacy/artifacts/job_index.json",
    }
    if path_text in inference_required:
        return "legacy_inference_required", True
    if path.suffix == ".keras":
        return "legacy_model_checkpoint", False
    if path.suffix in {".npy", ".parquet"}:
        return "training_only", False
    if path.suffix == ".ipynb":
        return "notebook_source", False
    if path_text.startswith("legacy/reports/") or path_text.startswith("reports/"):
        return "audit_or_evaluation_report", False
    if path.name.endswith("index.json"):
        return "legacy_inference_required", True
    return "metadata_or_reference", False


def inventory_paths() -> list[Path]:
    patterns = [
        "legacy/models/*.keras",
        "legacy/cache/*.npy",
        "legacy/cache/*.json",
        "legacy/artifacts/*.parquet",
        "legacy/artifacts/*.json",
        "legacy/reports/*",
        "legacy/*.ipynb",
        "training/notebooks/*.ipynb",
        "reports/*.json",
    ]
    paths: set[Path] = set()
    for pattern in patterns:
        paths.update(ROOT.glob(pattern))
    return sorted(paths, key=lambda path: path.as_posix())


def collect_artifact_inventory() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for path in inventory_paths():
        rel_path = path.relative_to(ROOT)
        role, required_for_inference = artifact_role(rel_path)
        row: dict[str, Any] = {
            "path": rel_path.as_posix(),
            "format": artifact_format(path),
            "owner": artifact_owner(rel_path),
            "artifact_role": role,
            "required_for_inference": required_for_inference,
            **file_metadata(path),
        }
        if path.exists() and path.suffix == ".npy":
            arr = np.load(path, mmap_mode="r")
            row["shape"] = list(arr.shape)
            row["dtype"] = str(arr.dtype)
        elif path.exists() and path.suffix == ".parquet":
            df = pd.read_parquet(path)
            row["rows"] = int(len(df))
            row["columns"] = list(df.columns)
        elif path.exists() and path.suffix == ".json":
            try:
                with path.open() as handle:
                    data = json.load(handle)
                row["json_type"] = type(data).__name__
                row["entries"] = len(data) if hasattr(data, "__len__") else None
            except Exception as exc:
                row["json_read_error"] = str(exc)
        rows.append(row)
    return rows


def collect_environment_snapshot() -> dict[str, Any]:
    packages = [
        "numpy",
        "pandas",
        "pyarrow",
        "scikit-learn",
        "tensorflow",
        "keras",
        "sentence-transformers",
        "jupyter",
        "ipykernel",
        "nbformat",
    ]
    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "python": {
            "version": platform.python_version(),
            "executable": sys.executable,
            "implementation": platform.python_implementation(),
        },
        "notebook_runtime": {
            "kernel": "python3",
            "execution_mode": "Jupyter-compatible notebook or nbconvert execute",
        },
        "packages": {name: package_version(name) for name in packages},
        "hardware_assumptions": {
            "platform": platform.platform(),
            "machine": platform.machine(),
            "processor": platform.processor() or "unknown",
            "cpu_count": os.cpu_count(),
            "gpu_required": False,
            "notes": "Phase 0 is an audit-only notebook. GPU is not required. Later training phases must record accelerator details separately.",
        },
        "random_seeds": {
            "validation_split_seed": SPLIT_SEED,
            "numpy_seed": "not_documented_in_legacy_training",
            "python_random_seed": "not_documented_in_legacy_training",
            "tensorflow_seed": "not_documented_in_legacy_training",
        },
    }


def grouped_split_indices(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    # Recreate the legacy notebook split exactly: GroupShuffleSplit grouped by profile_id.
    from sklearn.model_selection import GroupShuffleSplit

    splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SPLIT_SEED)
    groups = df["profile_id"].values
    return next(splitter.split(df, groups=groups))


def collect_dataset_snapshot() -> dict[str, Any]:
    jobs = pd.read_csv(LEGACY / "dataset" / "indotech_job_cleaned.csv")
    profiles = pd.read_csv(LEGACY / "dataset" / "techtalent_profile_cleaned.csv")
    pairs = pd.read_parquet(LEGACY / "artifacts" / "pairs.parquet")
    train_idx, val_idx = grouped_split_indices(pairs)

    cv_sample_paths = sorted(
        [
            path.relative_to(ROOT).as_posix()
            for pattern in ("**/*cv*.pdf", "**/*resume*.pdf", "**/*.docx")
            for path in LEGACY.glob(pattern)
        ]
    )

    return {
        "raw_jobs": {
            "path": "legacy/dataset/indotech_job_cleaned.csv",
            "rows": int(len(jobs)),
            "columns": list(jobs.columns),
            "missing_values": {column: int(jobs[column].isna().sum()) for column in jobs.columns},
            "language_distribution": jobs["language_signal"].fillna("MISSING").value_counts().to_dict(),
            "status_distribution": jobs["status"].fillna("MISSING").value_counts().to_dict(),
            "duplicate_job_id_count": int(jobs["job_id"].duplicated().sum()),
            "missing_job_id_count": int(jobs["job_id"].isna().sum()),
        },
        "raw_profiles": {
            "path": "legacy/dataset/techtalent_profile_cleaned.csv",
            "rows": int(len(profiles)),
            "columns": list(profiles.columns),
            "missing_values": {column: int(profiles[column].isna().sum()) for column in profiles.columns},
            "duplicate_profile_id_count": int(profiles["ID"].duplicated().sum()),
            "missing_profile_id_count": int(profiles["ID"].isna().sum()),
            "experience_distribution": profiles["Experience"].fillna("MISSING").value_counts().to_dict(),
        },
        "generated_pairs": {
            "path": "legacy/artifacts/pairs.parquet",
            "rows": int(len(pairs)),
            "columns": list(pairs.columns),
            "missing_values": {column: int(pairs[column].isna().sum()) for column in pairs.columns},
            "unique_profiles": int(pairs["profile_id"].nunique()),
            "unique_jobs": int(pairs["job_id"].nunique()),
            "duplicate_profile_job_pairs": int(pairs[["profile_id", "job_id"]].duplicated().sum()),
            "train_pairs": int(len(train_idx)),
            "validation_pairs": int(len(val_idx)),
            "split_strategy": {
                "type": "profile_id_group_shuffle",
                "validation_size": VAL_SIZE,
                "random_state": SPLIT_SEED,
            },
        },
        "cv_samples": {
            "count": len(cv_sample_paths),
            "paths": cv_sample_paths[:25],
            "note": "No CV benchmark sample files are present in the legacy snapshot." if not cv_sample_paths else "CV sample files found.",
        },
    }


def label_distribution(values: pd.Series | np.ndarray) -> dict[str, Any]:
    series = pd.Series(values)
    bands: dict[str, int] = {}
    for name, (lower, upper) in LABEL_BANDS.items():
        bands[name] = int(series.between(lower, upper, inclusive="both").sum())
    return {
        "count": int(series.count()),
        "min": round(float(series.min()), 6),
        "max": round(float(series.max()), 6),
        "mean": round(float(series.mean()), 6),
        "median": round(float(series.median()), 6),
        "std": round(float(series.std()), 6),
        "p25": round(float(series.quantile(0.25)), 6),
        "p75": round(float(series.quantile(0.75)), 6),
        "bands": bands,
    }


def skill_set(value: Any) -> set[str]:
    return {part.strip().lower() for part in str(value).split(",") if part.strip()}


def jaccard(a: Any, b: Any) -> float:
    left = skill_set(a)
    right = skill_set(b)
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


def cosine_similarity_rows(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    left_norm = np.linalg.norm(left, axis=1)
    right_norm = np.linalg.norm(right, axis=1)
    denom = left_norm * right_norm
    with np.errstate(divide="ignore", invalid="ignore"):
        scores = np.divide(np.sum(left * right, axis=1), denom, out=np.zeros(len(left), dtype=np.float32), where=denom != 0)
    return np.clip(scores, 0.0, 1.0)


def metric_block(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:
    errors = np.abs(y_true - y_pred)
    y_mean = float(np.mean(y_true))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_mean) ** 2))
    return {
        "mae": round(float(np.mean(errors)), 6),
        "accuracy_at_tolerance_0_15": round(float(np.mean(errors <= TOLERANCE)), 6),
        "r2": round(1.0 - ss_res / ss_tot, 6) if ss_tot else 0.0,
        "prediction_mean": round(float(np.mean(y_pred)), 6),
        "prediction_median": round(float(np.median(y_pred)), 6),
    }


def collect_baseline_metrics() -> dict[str, Any]:
    pairs = pd.read_parquet(LEGACY / "artifacts" / "pairs.parquet")
    train_idx, val_idx = grouped_split_indices(pairs)
    y_train = pairs["fit_score"].to_numpy(dtype=np.float32)[train_idx]
    y_val = pairs["fit_score"].to_numpy(dtype=np.float32)[val_idx]

    train_mean = float(np.mean(y_train))
    train_median = float(np.median(y_train))
    val_pairs = pairs.iloc[val_idx]
    skill_pred = val_pairs.apply(lambda row: jaccard(row["profile_skills"], row["job_skills"]), axis=1).to_numpy(dtype=np.float32)

    profile_embs = np.load(LEGACY / "cache" / "profile_embeddings.npy", mmap_mode="r")
    job_embs = np.load(LEGACY / "cache" / "job_embeddings.npy", mmap_mode="r")
    cosine_pred = cosine_similarity_rows(profile_embs[val_idx], job_embs[val_idx])

    with (LEGACY / "reports" / "model_card.json").open() as handle:
        model_card = json.load(handle)

    return {
        "source_model_card": "legacy/reports/model_card.json",
        "label_target": "fit_score",
        "label_source": "weak_label_rule_based",
        "prototype_gate_only": True,
        "current_metrics_exact_from_model_card": model_card.get("metrics", {}),
        "model_card_gate_passed": bool(model_card.get("gate_passed")),
        "label_distribution": {
            "overall": label_distribution(pairs["fit_score"]),
            "train": label_distribution(y_train),
            "validation": label_distribution(y_val),
        },
        "baseline_metrics": {
            "evaluation_split": "validation",
            "tolerance": TOLERANCE,
            "constant_mean": {"train_label_mean": round(train_mean, 6), **metric_block(y_val, np.full_like(y_val, train_mean))},
            "constant_median": {"train_label_median": round(train_median, 6), **metric_block(y_val, np.full_like(y_val, train_median))},
            "skill_jaccard_only": metric_block(y_val, skill_pred),
            "cosine_embedding_only": {
                "note": "Row-wise cosine(profile_embedding, job_embedding), clipped to [0, 1].",
                **metric_block(y_val, cosine_pred),
            },
        },
        "metric_notes": [
            "Current metrics are copied exactly from the legacy model card.",
            "The target label is a weak rule-based fit_score, not a human-reviewed production label.",
            "The legacy gate is prototype-only because R² is negative and no high-fit samples exist in the pair dataset.",
        ],
    }


def collect_risk_register() -> list[dict[str, str]]:
    return [
        {
            "risk": "Weak rule-based labels",
            "evidence": "Legacy fit_score is based on skill overlap and experience score, not recruiter or outcome labels.",
            "impact": "Model can learn the heuristic instead of real job-fit quality.",
            "next_control": "Define label schema and manual validation sample before new training.",
        },
        {
            "risk": "No high-fit samples",
            "evidence": "Current pair label maximum is below 0.65 and high band count is zero.",
            "impact": "Scores above strong-fit range cannot be calibrated from current data.",
            "next_control": "Build balanced pairs with low, medium, and high-fit examples.",
        },
        {
            "risk": "Possible data leakage",
            "evidence": "Legacy split groups by profile_id only; job family and duplicate text leakage are not fully audited.",
            "impact": "Validation metrics can overstate generalization.",
            "next_control": "Define stricter split rules in the pair-generation phase.",
        },
        {
            "risk": "Stale job artifacts",
            "evidence": "Legacy recommendation artifacts include static job_index and all_job_embeddings snapshots.",
            "impact": "Inference can rank inactive or backend-ineligible jobs if used directly.",
            "next_control": "Backend must provide candidate jobs and model should only score those candidates.",
        },
        {
            "risk": "Missing CV benchmark samples",
            "evidence": "No PDF/DOCX CV benchmark files are present in the legacy snapshot.",
            "impact": "ATS friendliness and extraction failure rates cannot be validated.",
            "next_control": "Create controlled CV parsing and ATS evaluation set before ATS scoring claims.",
        },
        {
            "risk": "Runtime incompatibility",
            "evidence": "Legacy custom layer/package loading is not documented as deployable outside the notebook.",
            "impact": "Model artifact can fail to load in API runtime.",
            "next_control": "Package model definition and record exact runtime versions before training v2.",
        },
    ]


## Step 0.1 — Artifact inventory

### Purpose
List every model, cache, pair dataset, report, and notebook artifact in the current snapshot.

### Required input
`legacy/models/`, `legacy/cache/`, `legacy/artifacts/`, `legacy/reports/`, `legacy/*.ipynb`, `training/notebooks/*.ipynb`, and `reports/*.json`.

### Action
Collect path, format, owner, artifact role, inference requirement flag, file size, hash, modification time, and lightweight shape or row metadata where available.

### Expected output
A reviewable artifact inventory table and JSON-ready inventory records.

### Verification
The inventory must include the legacy Keras model, embedding caches, pair dataset, job index, model card, legacy notebook, notebook-phase files, and generated audit reports.


In [8]:
artifact_inventory = collect_artifact_inventory()
artifact_inventory_df = pd.DataFrame(artifact_inventory)
summary_columns = ["path", "format", "owner", "artifact_role", "required_for_inference", "exists", "size_bytes"]
display(artifact_inventory_df[summary_columns])
print(f"Inventory item count: {len(artifact_inventory_df)}")
print("Inference-required artifacts:")
display(artifact_inventory_df.loc[artifact_inventory_df["required_for_inference"], ["path", "format", "sha256"]])


,path,format,owner,artifact_role,required_for_inference,exists,size_bytes
0,legacy/artifacts/job_index.json,json,legacy-model-training,legacy_inference_required,True,True,467334
1,legacy/artifacts/pairs.parquet,parquet,legacy-model-training,training_only,False,True,542922
2,legacy/bisakerja_model_training_FINAL.ipynb,ipynb,legacy-model-training,notebook_source,False,True,9073446
3,legacy/cache/all_job_embeddings.npy,npy,legacy-model-training,legacy_inference_required,True,True,3184256
4,legacy/cache/embedding_meta.json,json,legacy-model-training,metadata_or_reference,False,True,146
5,legacy/cache/job_embeddings.npy,npy,legacy-model-training,training_only,False,True,46080128
6,legacy/cache/profile_embeddings.npy,npy,legacy-model-training,training_only,False,True,46080128
7,legacy/models/checkpoint_best.keras,keras,legacy-model-training,legacy_model_checkpoint,False,True,4506778
8,legacy/models/model_jobfit_v1.keras,keras,legacy-model-training,legacy_inference_required,True,True,4506778
9,legacy/reports/eval_plot.png,png,legacy-model-training,audit_or_evaluation_report,False,True,49384


Inventory item count: 29
Inference-required artifacts:


,path,format,sha256
0,legacy/artifacts/job_index.json,json,3bd2130212eade240d16aa1aabb5e142fe7d4c5b3e0aad30db41891b61e81d2a
3,legacy/cache/all_job_embeddings.npy,npy,72cef692066d74d565b2395a51dc932bc8161e97e0880d2bddfbe08e702f0d0f
8,legacy/models/model_jobfit_v1.keras,keras,0d47e689392aa9f5e45a9efc1a71bedbbbecc1de4827ba29591423ee7c6aa17b


## Step 0.2 — Environment snapshot

### Purpose
Document Python version, notebook runtime, package versions, hardware assumptions, and random seeds that affect repeatability.

### Required input
Current Python runtime and installed package metadata.

### Action
Capture Python interpreter details, package versions, CPU/hardware assumptions, and known or missing random seed settings.

### Expected output
A deterministic environment snapshot that identifies both known seeds and missing legacy seed records.

### Verification
The snapshot must clearly mark undocumented seeds rather than inventing values.


In [9]:
environment_snapshot = collect_environment_snapshot()
print(json.dumps(environment_snapshot, indent=2))


{
  "generated_at_utc": "2026-06-01T06:56:27.942694+00:00",
  "python": {
    "version": "3.14.2",
    "executable": "/Users/macbookpro/.pyenv/versions/3.14.2/bin/python",
    "implementation": "CPython"
  },
  "notebook_runtime": {
    "kernel": "python3",
    "execution_mode": "Jupyter-compatible notebook or nbconvert execute"
  },
  "packages": {
    "numpy": "1.26.4",
    "pandas": "2.3.3",
    "pyarrow": "22.0.0",
    "scikit-learn": "1.8.0",
    "tensorflow": "not_installed",
    "keras": "not_installed",
    "sentence-transformers": "not_installed",
    "jupyter": "1.1.1",
    "ipykernel": "7.2.0",
    "nbformat": "5.10.4"
  },
  "hardware_assumptions": {
    "platform": "macOS-14.8.2-arm64-arm-64bit-Mach-O",
    "machine": "arm64",
    "processor": "arm",
    "cpu_count": 10,
    "gpu_required": false,
    "notes": "Phase 0 is an audit-only notebook. GPU is not required. Later training phases must record accelerator details separately."
  },
  "random_seeds": {
    "validation_

## Step 0.3 — Dataset snapshot

### Purpose
Count raw jobs, profiles, CV samples, generated pairs, missing values, language distribution, and duplicated identifiers.

### Required input
`legacy/dataset/indotech_job_cleaned.csv`, `legacy/dataset/techtalent_profile_cleaned.csv`, and `legacy/artifacts/pairs.parquet`.

### Action
Read the source datasets and generated pairs, count rows, nulls, duplicate identifiers, language/status distribution, split counts, and available CV benchmark samples.

### Expected output
Dataset summary tables that expose row counts, missing values, duplicated identifiers, and CV benchmark availability.

### Verification
Counts must match the legacy snapshot before later phases create new pairs or labels.


In [10]:
dataset_snapshot = collect_dataset_snapshot()

row_summary = pd.DataFrame([
    {"dataset": "raw_jobs", "rows": dataset_snapshot["raw_jobs"]["rows"], "identifier_duplicates": dataset_snapshot["raw_jobs"]["duplicate_job_id_count"], "missing_identifier": dataset_snapshot["raw_jobs"]["missing_job_id_count"]},
    {"dataset": "raw_profiles", "rows": dataset_snapshot["raw_profiles"]["rows"], "identifier_duplicates": dataset_snapshot["raw_profiles"]["duplicate_profile_id_count"], "missing_identifier": dataset_snapshot["raw_profiles"]["missing_profile_id_count"]},
    {"dataset": "generated_pairs", "rows": dataset_snapshot["generated_pairs"]["rows"], "identifier_duplicates": dataset_snapshot["generated_pairs"]["duplicate_profile_job_pairs"], "missing_identifier": 0},
    {"dataset": "cv_samples", "rows": dataset_snapshot["cv_samples"]["count"], "identifier_duplicates": 0, "missing_identifier": 0},
])
display(row_summary)

display(pd.DataFrame([dataset_snapshot["generated_pairs"]]).drop(columns=["columns", "missing_values"]))
print("Job language distribution:")
display(pd.Series(dataset_snapshot["raw_jobs"]["language_distribution"], name="count").to_frame())
print("Job status distribution:")
display(pd.Series(dataset_snapshot["raw_jobs"]["status_distribution"], name="count").to_frame())
print(dataset_snapshot["cv_samples"]["note"])


,dataset,rows,identifier_duplicates,missing_identifier
0,raw_jobs,2073,0,0
1,raw_profiles,69929,0,0
2,generated_pairs,30000,0,0
3,cv_samples,0,0,0


,path,rows,unique_profiles,unique_jobs,duplicate_profile_job_pairs,train_pairs,validation_pairs,split_strategy
0,legacy/artifacts/pairs.parquet,30000,3000,2071,0,25500,4500,"{'type': 'profile_id_group_shuffle', 'validation_size': 0.15, 'random_state': 42}"


Job language distribution:


,count
EN,1426
UNKNOWN,602
MIXED,23
ID,22


Job status distribution:


,count
ACTIVE,2072
EXPIRED,1


No CV benchmark sample files are present in the legacy snapshot.


## Step 0.4 — Baseline metric capture

### Purpose
Copy existing metrics exactly as reported and mark any metric that came from weak labels or prototype-only gates.

### Required input
`legacy/reports/model_card.json`, `legacy/artifacts/pairs.parquet`, and embedding caches used for simple baseline comparisons.

### Action
Read the legacy model card metrics exactly, recreate validation-split baseline metrics, summarize label distribution, and flag weak-label/prototype-gate limitations.

### Expected output
A metrics snapshot showing current model-card metrics, constant baselines, skill-overlap baseline, cosine baseline, and weak-label notes.

### Verification
The model-card metrics must remain exact copies. Baselines must use train-only constants and validation-only evaluation.


In [11]:
baseline_snapshot = collect_baseline_metrics()
print("Current metrics copied exactly from legacy/reports/model_card.json:")
print(json.dumps(baseline_snapshot["current_metrics_exact_from_model_card"], indent=2))

print("Label distribution:")
display(pd.DataFrame(baseline_snapshot["label_distribution"]).T)

print("Baseline metrics:")
display(pd.DataFrame(baseline_snapshot["baseline_metrics"]).T)

print("Metric notes:")
for note in baseline_snapshot["metric_notes"]:
    print(f"- {note}")


Current metrics copied exactly from legacy/reports/model_card.json:
{
  "mae": 0.0844,
  "accuracy": 0.802,
  "r2": -0.0038
}
Label distribution:


,count,min,max,mean,median,std,p25,p75,bands
overall,30000,0.03,0.5545,0.206089,0.21,0.103143,0.12,0.3,"{'low': 29302, 'medium': 557, 'high': 0}"
train,25500,0.03,0.5333,0.206502,0.21,0.102982,0.12,0.3,"{'low': 24902, 'medium': 473, 'high': 0}"
validation,4500,0.03,0.5545,0.20375,0.21,0.104032,0.12,0.3,"{'low': 4400, 'medium': 84, 'high': 0}"


Baseline metrics:


,train_label_mean,mae,accuracy_at_tolerance_0_15,r2,prediction_mean,prediction_median,train_label_median,note
evaluation_split,validation,validation,validation,validation,validation,validation,validation,validation
tolerance,0.15,0.15,0.15,0.15,0.15,0.15,0.15,0.15
constant_mean,0.206502,0.085552,0.800667,-0.0007,0.206502,0.206502,NaN,NaN
constant_median,NaN,0.084368,0.801778,-0.00361,0.21,0.21,0.21,NaN
skill_jaccard_only,NaN,0.192715,0.339778,-3.435032,0.011186,0.0,NaN,NaN
cosine_embedding_only,NaN,0.292163,0.148,-8.53008,0.495397,0.490751,NaN,"Row-wise cosine(profile_embedding, job_embedding), clipped to [0, 1]."


Metric notes:
- Current metrics are copied exactly from the legacy model card.
- The target label is a weak rule-based fit_score, not a human-reviewed production label.
- The legacy gate is prototype-only because R² is negative and no high-fit samples exist in the pair dataset.


## Step 0.5 — Risk register

### Purpose
Write known risks before new work starts, including label quality, data leakage, missing high-fit samples, stale jobs, and runtime incompatibilities.

### Required input
Artifact inventory, environment snapshot, dataset snapshot, baseline metric snapshot, and the known training gaps documented in `GAP_MODEL_TRAINING.md`.

### Action
Create a risk register with evidence, impact, and the next control for each blocker. Write the complete Phase 0 snapshot report to `reports/phase_00_reproducibility_snapshot.json`.

### Expected output
A visible risk register and reproducibility JSON report that later phases can reference.

### Verification
The report must include inventory, environment, dataset, metrics, risks, and acceptance status.


In [12]:
risk_register = collect_risk_register()
phase0_snapshot = {
    "schema_version": "phase-00-reproducibility-snapshot-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_snapshot": "legacy",
    "artifact_inventory": artifact_inventory,
    "environment_snapshot": environment_snapshot,
    "dataset_snapshot": dataset_snapshot,
    "baseline_metric_snapshot": baseline_snapshot,
    "risk_register": risk_register,
    "acceptance": {
        "inventory_covers_required_training_artifacts": True,
        "baseline_metrics_recorded_before_new_experiments": True,
        "known_blockers_visible_before_phase_1": True,
    },
}

REPORTS.mkdir(parents=True, exist_ok=True)
with PHASE0_REPORT.open("w") as handle:
    json.dump(phase0_snapshot, handle, indent=2)
    handle.write("\n")

risk_df = pd.DataFrame(risk_register)
display(risk_df)
print(f"Wrote {PHASE0_REPORT.relative_to(ROOT)}")
print(json.dumps(phase0_snapshot["acceptance"], indent=2))


,risk,evidence,impact,next_control
0,Weak rule-based labels,"Legacy fit_score is based on skill overlap and experience score, not recruiter or outcome labels.",Model can learn the heuristic instead of real job-fit quality.,Define label schema and manual validation sample before new training.
1,No high-fit samples,Current pair label maximum is below 0.65 and high band count is zero.,Scores above strong-fit range cannot be calibrated from current data.,"Build balanced pairs with low, medium, and high-fit examples."
2,Possible data leakage,Legacy split groups by profile_id only; job family and duplicate text leakage are not fully audited.,Validation metrics can overstate generalization.,Define stricter split rules in the pair-generation phase.
3,Stale job artifacts,Legacy recommendation artifacts include static job_index and all_job_embeddings snapshots.,Inference can rank inactive or backend-ineligible jobs if used directly.,Backend must provide candidate jobs and model should only score those candidates.
4,Missing CV benchmark samples,No PDF/DOCX CV benchmark files are present in the legacy snapshot.,ATS friendliness and extraction failure rates cannot be validated.,Create controlled CV parsing and ATS evaluation set before ATS scoring claims.
5,Runtime incompatibility,Legacy custom layer/package loading is not documented as deployable outside the notebook.,Model artifact can fail to load in API runtime.,Package model definition and record exact runtime versions before training v2.


Wrote reports/phase_00_reproducibility_snapshot.json
{
  "inventory_covers_required_training_artifacts": true,
  "baseline_metrics_recorded_before_new_experiments": true,
  "known_blockers_visible_before_phase_1": true
}


## Acceptance criteria

- [x] Inventory covers all required training artifacts.
- [x] Baseline metrics are recorded before new experiments.
- [x] Known blockers are visible before Phase 1 starts.

## Phase notes

- Legacy metrics remain prototype-only because `fit_score` is weak-label based and the current R² is negative.
- No high-fit pair labels are present in the current generated pair dataset.
- No CV benchmark samples are present, so ATS friendliness cannot be production-validated yet.
- Static job artifacts must not own final backend job hydration.
